In [12]:
import sys
!conda install --yes --prefix {sys.prefix} numpy
!conda install --yes --prefix {sys.prefix} pandas
!conda install --yes --prefix {sys.prefix} scikit-learn

Jupyter detected...
2 channel Terms of Service accepted
Channels:
 - defaults
Platform: linux-64
doneecting package metadata (repodata.json): - 
doneing environment: - 

# All requested packages already installed.

Jupyter detected...
2 channel Terms of Service accepted
Channels:
 - defaults
Platform: linux-64
doneecting package metadata (repodata.json): - 
doneing environment: / 

# All requested packages already installed.

Jupyter detected...
2 channel Terms of Service accepted
Channels:
 - defaults
Platform: linux-64
doneecting package metadata (repodata.json): - 
doneing environment: / 

# All requested packages already installed.



In [1]:
import json
import ast
import numpy as np
import pandas as pd
from sklearn.preprocessing import TargetEncoder, StandardScaler

In [2]:
training_dataframe = pd.read_csv('Dataset/labelled_training_data.csv')
testing_dataframe = pd.read_csv('Dataset/labelled_testing_data.csv')
validation_dataframe = pd.read_csv('Dataset/labelled_validation_data.csv')

raw_training_data = training_dataframe.values
raw_testing_data = testing_dataframe.values
raw_validation_data = validation_dataframe.values

training_dataframe.head()

,timestamp,processId,threadId,parentProcessId,userId,mountNamespace,processName,hostName,eventId,eventName,stackAddresses,argsNum,returnValue,args,sus,evil
0,1809.495787,381,7337,1,100,4026532231,close,ip-10-100-1-120,157,prctl,"[140662171848350, 11649800180280676]",5,0,"[{'name': 'option', 'type': 'int', 'value': 'P...",1,0
1,1809.495832,381,7337,1,100,4026532231,close,ip-10-100-1-120,3,close,[140662171777451],1,0,"[{'name': 'fd', 'type': 'int', 'value': 19}]",1,0
2,1809.495921,381,7337,1,100,4026532231,close,ip-10-100-1-120,1010,sched_process_exit,[],0,0,[],1,0
3,1894.139651,7347,7347,7341,0,4026531840,sh,ip-10-100-1-120,21,access,[],2,-2,"[{'name': 'pathname', 'type': 'const char*', '...",1,0
4,1894.142127,7347,7347,7341,0,4026531840,sh,ip-10-100-1-120,1005,security_file_open,"[139778263990104, 139778263906698]",4,0,"[{'name': 'pathname', 'type': 'const char*', '...",1,0


As adviced by the authors of the beth dataset's paper, we applied these transformation:

- ProcessId and ParentprocessId: 0 if it is [0,1,2] otherwise 1
- UserId: 0 if id is less than 1000 otherwise 1
- MountNameSpace: 0 if it is equal to 4026531840 otherwise 1
- ReturnValue: 0 if it is 0, 1 if it is positive and 2 if it is negative     
- Drop timestamp column since not time series related
- Drop eventName as uniquely mapped to eventId
- Encode processName, threadId and eventId using target encoding
- Drop stackAddresses
- Drop HostName
- Args column contains a list of maximum 5 dictionaries, each disctionary contains three elements ({'name': 'dev', 'type': 'dev_t', 'value': 211812353}). We created 15 new columns in each datset, and assigns each element from the dictionaries to its respective new column. After which apply target encoding.

There is no need to normalize or scale the data as the magnitude is not relevant for the numeric data with the exception of ArgsNum which I encoded using scalar (which I commented out in this case).

In [3]:
# Function to split the string column into list of dictionaries and create new columns
def split_and_expand(row):
    if pd.isna(row):
        return pd.Series([None] * 15)

    dicts = ast.literal_eval(row)
    result = {'name_{}'.format(i+1): None for i in range(5)}
    result.update({'type_{}'.format(i+1): None for i in range(5)})
    result.update({'value_{}'.format(i+1): None for i in range(5)})

    for i, d in enumerate(dicts):
        if i >= 5:  # We only want to process the first 5 dictionaries
            break
        result['name_{}'.format(i+1)] = d.get('name')
        result['type_{}'.format(i+1)] = d.get('type')
        result['value_{}'.format(i+1)] = d.get('value')

    return pd.Series(result)

In [4]:
def apply_transformations(data, encoder, scaler, is_training=False):

    # Define column names
    columns = ['timestamp', 'processId', 'threadId', 'parentProcessId', 'userId', 
               'mountNamespace', 'processName', 'hostName', 'eventId', 'eventName', 
               'stackAddresses', 'argsNum', 'returnValue', 'args', 'sus', 'evil']

    # Convert to DataFrame for easier column manipulation
    df = pd.DataFrame(data, columns=columns)
    
    # Transform ProcessId
    df['processId'] = np.where(np.isin(df['processId'], [0, 1, 2]), 0, 1)
    # Transform ParentProcessId
    df['parentProcessId'] = np.where(np.isin(df['parentProcessId'], [0, 1, 2]), 0, 1)
    # Transform UserId
    df['userId'] = np.where(df['userId'] < 1000, 0, 1)
    # Transform MountNameSpace
    df['mountNamespace'] = np.where(df['mountNamespace'] == 4026531840, 0, 1)
    # Transform ReturnValue
    df['returnValue'] = np.where(df['returnValue'] == 0, 0,
                                np.where(df['returnValue'] > 0, 1, 2))
    
    # Store Args Column before dropping
    args_column = df['args'].copy()

    # Normalise Args Num
    # Reshape the 'argsNum' column to a 2D array before applying the scaler
    df['argsNum'] = scaler.fit_transform(df['argsNum'].values.reshape(-1, 1))
    
    # Drop columns by name
    cols_to_drop = ['timestamp', 'stackAddresses', 'args', 'eventName', 'hostName']
    df_filtered = df.drop(columns=cols_to_drop)
    
    # Apply split_and_expand to the stored args column
    expanded_args = args_column.apply(split_and_expand)
    
    # Join the expanded columns with the DataFrame
    df_data = df_filtered.join(expanded_args)

    categorical_cols = [col for col in df_data.columns if 'name_' in str(col) or 'type_' in str(col) or 'value_' in str(col)] + ['processName', 'threadId', 'eventId']
    for col in categorical_cols:
        # Replace NaN with 'unknown'
        df_data[col] = df_data[col].fillna('unknown')
        # Convert all values in categorical columns to strings
        df_data[col] = df_data[col].astype(str)

    # Fit the encoder on the training set (if not already fitted)
    if is_training:
        encoder.fit(df_data[categorical_cols], df_data['sus'].astype(int))
        
    # Use the existing encoder for validation/test data
    encoded_data = encoder.transform(df_data[categorical_cols])
    df_encoded = pd.DataFrame(encoded_data, columns=encoder.get_feature_names_out(categorical_cols), index=df_data.index)
    
    # Drop original categorical columns and concatenate encoded columns
    df_data = df_data.drop(columns=categorical_cols)
    df_data = pd.concat([df_data, df_encoded], axis=1)
    
    return df_data

In [5]:
c_encoder = TargetEncoder()
scaler = StandardScaler() # Not used by I lazy adapt the code 

# Apply transformations to the datasets
transformed_training_data = apply_transformations(raw_training_data, c_encoder, scaler, True)
transformed_testing_data = apply_transformations(raw_testing_data, c_encoder, scaler)
transformed_validation_data = apply_transformations(raw_validation_data, c_encoder, scaler)

/tmp/ipykernel_14197/1419985687.py:43: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_data[col] = df_data[col].fillna('unknown')
/tmp/ipykernel_14197/1419985687.py:43: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_data[col] = df_data[col].fillna('unknown')
/tmp/ipykernel_14197/1419985687.py:43: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', Tr

In [6]:
# Transformed Dataset (including labels)
pd.set_option("display.max_columns", None)
labelled_training_df = pd.DataFrame(transformed_training_data)
print(labelled_training_df.tail())

        processId  parentProcessId  userId  mountNamespace   argsNum  \
763139          1                1       0               0 -1.246980   
763140          1                1       0               0 -0.501215   
763141          1                1       0               0 -1.992744   
763142          1                0       0               0 -1.992744   
763143          1                0       0               0 -1.246980   

        returnValue sus evil    name_1    name_2    name_3    name_4  \
763139            0   0    0  0.000000  0.001142  0.002022  0.002133   
763140            0   0    0  0.000000  0.000000  0.002022  0.002133   
763141            0   0    0  0.023907  0.001142  0.002022  0.002133   
763142            0   0    0  0.023907  0.001142  0.002022  0.002133   
763143            0   0    0  0.001354  0.001142  0.002022  0.002133   

          name_5    type_1    type_2    type_3    type_4    type_5   value_1  \
763139  0.001644  0.001350  0.001142  0.002022  0.0021

Split the data and labels

In [7]:
# Get the labels by selecting the 9th column (index 8)
training_labels = transformed_training_data.iloc[:, 6]
testing_labels = transformed_testing_data.iloc[:, 6]
validation_labels = transformed_validation_data.iloc[:, 6]

# Remove the columns 8 and 9 (labels) from the features
training_data = transformed_training_data.drop(columns=[transformed_training_data.columns[6], transformed_training_data.columns[7]])
testing_data = transformed_testing_data.drop(columns=[transformed_testing_data.columns[6], transformed_testing_data.columns[7]])
validation_data = transformed_validation_data.drop(columns=[transformed_validation_data.columns[6], transformed_validation_data.columns[7]])

In [8]:
# Transformed Data (without labels)
pd.set_option("display.max_columns", None)
training_df = pd.DataFrame(training_data)
print(training_df.tail())

        processId  parentProcessId  userId  mountNamespace   argsNum  \
763139          1                1       0               0 -1.246980   
763140          1                1       0               0 -0.501215   
763141          1                1       0               0 -1.992744   
763142          1                0       0               0 -1.992744   
763143          1                0       0               0 -1.246980   

        returnValue    name_1    name_2    name_3    name_4    name_5  \
763139            0  0.000000  0.001142  0.002022  0.002133  0.001644   
763140            0  0.000000  0.000000  0.002022  0.002133  0.001644   
763141            0  0.023907  0.001142  0.002022  0.002133  0.001644   
763142            0  0.023907  0.001142  0.002022  0.002133  0.001644   
763143            0  0.001354  0.001142  0.002022  0.002133  0.001644   

          type_1    type_2    type_3    type_4    type_5   value_1   value_2  \
763139  0.001350  0.001142  0.002022  0.002133  

Saving the files to CSV

In [9]:
labelled_training_df.to_csv("Dataset/Transformed/Normalised/labelled_transformed_training_data.csv", index=False)
training_df.to_csv("Dataset/Transformed/Normalised/transformed_training_data.csv", index=False)

labelled_testing_df = pd.DataFrame(transformed_testing_data)
labelled_testing_df.to_csv("Dataset/Transformed/Normalised/labelled_transformed_testing_data.csv", index=False)
testing_df = pd.DataFrame(testing_data)
testing_df.to_csv("Dataset/Transformed/Normalised/transformed_testing_data.csv", index=False)

labelled_validation_df = pd.DataFrame(transformed_validation_data)
labelled_validation_df.to_csv("Dataset/Transformed/Normalised/labelled_transformed_validation_data.csv", index=False)
validation_df = pd.DataFrame(validation_data)
validation_df.to_csv("Dataset/Transformed/Normalised/transformed_validation_data.csv", index=False)